# Stroke unit locations

Update the hospital reference with the units new to Samuel-3.

For a given point, for example a stroke unit location, convert the coordinates from latitude/longitude to British National Grid units and find which LSOA the point lies in.

In this notebook, find this data for each "new" stroke unit:
+ 'SSNAP name'
+ 'Stroke Team'
+ 'Postcode'
+ 'long'
+ 'lat'
+ 'Easting'
+ 'Northing'
+ 'Country'
+ 'Strategic Clinical Network'
+ 'Health Board / Trust'
+ 'hospital_city'

Method:
1. Find the postcode, containing hospital, and longitude and latitude from Google Maps.
2. Use geopandas to convert lat/long to British National Grid Easting/Northing.
3. Note which city/town/village each new unit is in.
4. Copy the other geographical data from appropriate nearby stroke units that share a health board, SCN, country etc.

## Unit matching

Assume that some "new" units have the same geographical data as old ones:
+ 'Bristol and Weston SU' replaces 'North Bristol Hospitals', i.e. identical location
+ 'East Kent HASU' replaces 'William Harvey Hospital', i.e. identical location

Have to set up new location data for:
+ 'Royal Glamorgan'
+ 'Midland Metropolitan University Hospital (MMUH) - Acute Stroke Services'
+ 'Cheltenham & Gloucester Hospitals'

... but assume that these share broader regional info with a nearby unit:
+ 'Royal Glamorgan' is the same region as 'Princess of Wales Hospital'
+ 'Midland Metropolitan University Hospital (MMUH) - Acute Stroke Services' is the same region as 'Sandwell District Hospital'
+ 'Cheltenham & Gloucester Hospitals' is the same region as 'Gloucestershire Royal Hospital'

## Code setup

In [1]:
import pandas as pd
import geopandas as gpd
import os

## Load Samuel-2 hospital data

Check which information we need to gather for the new hospitals:

In [2]:
df_units = pd.read_csv(os.path.join('data', 'stroke_hospitals_2022.csv'))
columns_original = df_units.columns
df_units = df_units.set_index('Postcode')

In [3]:
df_units.columns

Index(['Hospital_name', 'Use_IVT', 'Use_MT', 'Use_MSU', 'Country',
       'Strategic Clinical Network', 'Health Board / Trust', 'Stroke Team',
       'SSNAP name', 'Admissions 21/22', 'Thrombolysis', 'ivt_rate', 'Easting',
       'Northing', 'long', 'lat', 'Neuroscience',
       '30 England Thrombectomy Example', 'hospital_city', 'Notes'],
      dtype='object')

Drop non-geographical columns from the Samuel-2 data:

In [4]:
cols_to_drop = [
    'Use_IVT', 'Use_MT', 'Use_MSU', 'Admissions 21/22', 'Thrombolysis', 'ivt_rate',
    'Neuroscience', '30 England Thrombectomy Example', 'Notes'
]
df_units = df_units.drop(cols_to_drop, axis='columns')

In [5]:
df_units.head(3)

,Hospital_name,Country,Strategic Clinical Network,Health Board / Trust,Stroke Team,SSNAP name,Easting,Northing,long,lat,hospital_city
Postcode,,,,,,,,,,,
RM70AG,RM70AG,England,London SCN,Barking,Havering and Redbridge University Hospitals N...,Queens Hospital Romford HASU,551118,187780,0.179031,51.568647,Romford
E11BB,E11BB,England,London SCN,Barts Health NHS Trust,The Royal London Hospital,Royal London Hospital HASU,534829,181798,-0.058133,51.519018,Royal London
SW66SX,SW66SX,England,London SCN,Imperial College Healthcare NHS Trust,"Charing Cross Hospital, London",Charing Cross Hospital HASU,524226,176487,-0.212736,51.473717,Charing Cross


## Set up data


Postcodes, hospitals and coordinates in lat/long of a few points:

In [6]:
# Coordinates of units new to Samuel-3:
dict_latlong = {
    'CF728XR': ['Royal Glamorgan Hospital', 'Royal Glamorgan', 'Ynysmaerdy', 51.54828400503419, -3.391390931107454],
    'B662QT': ['Midland Metropolitan University Hospital (MMUH)', 'Midland Metropolitan University Hospital (MMUH) - Acute Stroke Services', 'Smethwick', 52.49084355816159, -1.9498965292101011],
    'GL537AN': ['Cheltenham General Hospital', 'Cheltenham & Gloucester Hospitals', 'Cheltenham', 51.89324094153232, -2.0715041230465876],
}

In [7]:
df_points = pd.DataFrame(dict_latlong.values(), index=dict_latlong.keys(),
                         columns=['Stroke Team', 'SSNAP name', 'hospital_city', 'lat', 'long'])
df_points.index.name = 'Postcode'

In [8]:
df_points

,Stroke Team,SSNAP name,hospital_city,lat,long
Postcode,,,,,
CF728XR,Royal Glamorgan Hospital,Royal Glamorgan,Ynysmaerdy,51.548284,-3.391391
B662QT,Midland Metropolitan University Hospital (MMUH),Midland Metropolitan University Hospital (MMUH...,Smethwick,52.490844,-1.949897
GL537AN,Cheltenham General Hospital,Cheltenham & Gloucester Hospitals,Cheltenham,51.893241,-2.071504


## Convert lat, long to geopandas points

Coordinate reference systems:
+ "EPSG:4326" for latitude and longitude
+ "EPSG:27700" for British National Grid

In [9]:
g = gpd.points_from_xy(df_points['long'], df_points['lat'], crs='EPSG:4326')
gdf = gpd.GeoDataFrame(df_points, geometry=g)

In [10]:
gdf

,Stroke Team,SSNAP name,hospital_city,lat,long,geometry
Postcode,,,,,,
CF728XR,Royal Glamorgan Hospital,Royal Glamorgan,Ynysmaerdy,51.548284,-3.391391,POINT (-3.39139 51.54828)
B662QT,Midland Metropolitan University Hospital (MMUH),Midland Metropolitan University Hospital (MMUH...,Smethwick,52.490844,-1.949897,POINT (-1.9499 52.49084)
GL537AN,Cheltenham General Hospital,Cheltenham & Gloucester Hospitals,Cheltenham,51.893241,-2.071504,POINT (-2.0715 51.89324)


## Convert lat, long to British National Grid coordinates

Use the geopandas `.to_crs()` function:

In [11]:
df_bng = gdf.to_crs('EPSG:27700').get_coordinates()
df_bng = df_bng.rename(columns={'x': 'Easting', 'y': 'Northing'})
# Round values:
df_bng = df_bng.round(0).astype(int)

In [12]:
gdf = pd.concat((gdf, df_bng), axis='columns')

In [13]:
gdf

,Stroke Team,SSNAP name,hospital_city,lat,long,geometry,Easting,Northing
Postcode,,,,,,,,
CF728XR,Royal Glamorgan Hospital,Royal Glamorgan,Ynysmaerdy,51.548284,-3.391391,POINT (-3.39139 51.54828),303619,184183
B662QT,Midland Metropolitan University Hospital (MMUH),Midland Metropolitan University Hospital (MMUH...,Smethwick,52.490844,-1.949897,POINT (-1.9499 52.49084),403499,288103
GL537AN,Cheltenham General Hospital,Cheltenham & Gloucester Hospitals,Cheltenham,51.893241,-2.071504,POINT (-2.0715 51.89324),395176,221634


## Link in other region info

The "Hospital_name" column is just a copy of the postcode:

In [14]:
gdf['Hospital_name'] = gdf.index

From the other nearby units, copy over the following: Country, Strategic Clinical Network, Health Board / Trust

In [15]:
cols_region = ['Country', 'Strategic Clinical Network', 'Health Board / Trust']

unit_matches = {
    'CF728XR': 'CF311RQ', # 'Royal Glamorgan': 'Princess Of Wales Hospital',
    'B662QT': 'B714HJ', # 'Midland Metropolitan University Hospital (MMUH) - Acute Stroke Services': 'Sandwell District Hospital',
    'GL537AN': 'GL13NN', # 'Cheltenham & Gloucester Hospitals': 'Gloucestershire Royal Hospital'
}

for new_unit, old_unit in unit_matches.items():
    gdf.loc[new_unit, cols_region] = df_units.loc[old_unit, cols_region].copy()

## Gather all unit info

Gather the info for all units, even ones that were only before Samuel-3.

In [16]:
gdf = gdf[df_units.columns]

In [17]:
df_units = pd.concat((df_units.reset_index(), gdf.reset_index()),
                     axis='rows', ignore_index=True)

Copy over the data for new units that share old unit locations:

In [18]:
df_units = df_units.set_index('SSNAP name')
df_units.loc['Bristol and Weston SU'] = df_units.loc['North Bristol Hospitals']
df_units.loc['East Kent HASU'] = df_units.loc['William Harvey Hospital']
df_units = df_units.reset_index()

In [19]:
df_units.tail()

,SSNAP name,Postcode,Hospital_name,Country,Strategic Clinical Network,Health Board / Trust,Stroke Team,Easting,Northing,long,lat,hospital_city
142,Royal Glamorgan,CF728XR,CF728XR,Wales,Wales,Wales,Royal Glamorgan Hospital,303619,184183,-3.391391,51.548284,Ynysmaerdy
143,Midland Metropolitan University Hospital (MMUH...,B662QT,B662QT,England,West Midlands SCN,Sandwell and West Birmingham Hospitals NHS Trust,Midland Metropolitan University Hospital (MMUH),403499,288103,-1.949897,52.490844,Smethwick
144,Cheltenham & Gloucester Hospitals,GL537AN,GL537AN,England,South West SCN,Gloucestershire Hospitals NHS Foundation Trust,Cheltenham General Hospital,395176,221634,-2.071504,51.893241,Cheltenham
145,Bristol and Weston SU,BS105NB,BS105NB,England,South West SCN,North Bristol NHS Trust,North Bristol Hospital (Southmead),358934,177759,-2.592962,51.497270,North Bristol
146,East Kent HASU,TN240LZ,TN240LZ,England,South East SCN,East Kent Hospitals University NHS Foundation ...,"William Harvey Hospital, Ashford",604090,142068,0.916209,51.141489,Ashford


## Find LSOA containing points

Load in LSOA shapefile:

In [20]:
path_to_lsoa = '~/samuel_book/geography_data/data_geojson/ons_data'
file_lsoa = 'LSOA_Dec_2011_Boundaries_Generalised_Clipped_BGC_EW_V3.geojson'
p = os.path.join(path_to_lsoa, file_lsoa)

gdf_lsoa = gpd.read_file(p)
# gdf = gdf.set_index('FID')
gdf_lsoa = gdf_lsoa.set_index('LSOA11CD')

gdf_lsoa.head()

,FID,LSOA11NM,LSOA11NMW,BNG_E,BNG_N,LONG,LAT,GlobalID,geometry
LSOA11CD,,,,,,,,,
E01000001,1,City of London 001A,City of London 001A,532129,181625,-0.097060,51.51810,283b0ead-f8fc-40b6-9a79-1ddd7e5c0758,"POLYGON ((532105.092 182011.23, 532162.491 181..."
E01000002,2,City of London 001B,City of London 001B,532480,181699,-0.091970,51.51868,ddce266b-7825-428c-9e0a-df66b0179a55,"POLYGON ((532634.497 181926.016, 532619.141 18..."
E01000003,3,City of London 001C,City of London 001C,532245,182036,-0.095230,51.52176,c45e358e-a794-485a-bf76-d96e5d458ea4,"POLYGON ((532135.138 182198.131, 532158.25 182..."
E01000005,4,City of London 001E,City of London 001E,533581,181265,-0.076280,51.51452,4ddaf5e4-e47f-4312-89a0-923ffec028a6,"POLYGON ((533808.018 180767.774, 533649.037 18..."
E01000006,5,Barking and Dagenham 016A,Barking and Dagenham 016A,544994,184276,0.089318,51.53876,1c04702a-b662-4cfc-aab9-2c3e0f2d5e29,"POLYGON ((545122.049 184314.931, 545271.849 18..."


Convert unit lat, long to geopandas points:

Coordinate reference systems:
+ "EPSG:4326" for latitude and longitude
+ "EPSG:27700" for British National Grid

In [21]:
g = gpd.points_from_xy(df_units['long'], df_units['lat'], crs='EPSG:4326')
gdf = gpd.GeoDataFrame(df_units, geometry=g)

Find which LSOA contains each point. Check that the coordinate systems match between points and LSOA geometry!

In [22]:
for point in gdf.index:
    p = gdf.to_crs('EPSG:27700').loc[point, 'geometry']
    s = gdf_lsoa.contains(p)
    lsoa_here = s[s].index.values[0]
    gdf.loc[point, 'LSOA11CD'] = lsoa_here

In [23]:
gdf.head(3)

,SSNAP name,Postcode,Hospital_name,Country,Strategic Clinical Network,Health Board / Trust,Stroke Team,Easting,Northing,long,lat,hospital_city,geometry,LSOA11CD
0,Queens Hospital Romford HASU,RM70AG,RM70AG,England,London SCN,Barking,Havering and Redbridge University Hospitals N...,551118,187780,0.179031,51.568647,Romford,POINT (0.17903 51.56865),E01002248
1,Royal London Hospital HASU,E11BB,E11BB,England,London SCN,Barts Health NHS Trust,The Royal London Hospital,534829,181798,-0.058133,51.519018,Royal London,POINT (-0.05813 51.51902),E01004322
2,Charing Cross Hospital HASU,SW66SX,SW66SX,England,London SCN,Imperial College Healthcare NHS Trust,"Charing Cross Hospital, London",524226,176487,-0.212736,51.473717,Charing Cross,POINT (-0.21274 51.47372),E01001906


Copy LSOA codes into main units reference:

In [24]:
df_units = pd.concat((df_units, gdf['LSOA11CD']), axis='columns')

Save a copy:

In [25]:
df_units.to_csv(os.path.join('data', 'stroke_hospitals_2025_geog.csv'), index=False)